# 🎯 Workshop Day 3: Evaluation & Optimization

**Agentic RAG Workshop** | Day 3 of 3

---

### 🎯 
1. ** RAG** — RAGAS (4 metrics)
2. ** Eval Dataset** — Gemini
3. ** Agent** — Tool Selection, LLM-as-Judge
4. ** Pipeline** — A/B Experiment
5. **Prompt Engineering** — Instruction 
6. **Capstone Project** — 3 ⭐

### 🗺️ 3 

```
Day 1 (Data) Day 2 (Agent) Day 3 (Evaluation)
───────────── ───────────── ─────────────────
Raw → Chunk → Agent + Tool → (3.1-3.2)
Embed → VectorDB RAG Agent → Agent (3.3)
 → Retrieve Multi-Agent → (3.4-3.5)
 Agentic RAG Capstone ⭐ (3.6)
```

> 💡 "" — 

## 📦 Section 0: Dependencies

In [ ]:
%%time
pip3 install python-pptx 2>&1 | tail -3 install -q ragas datasets google-adk google-genai \
 sentence-transformers qdrant-client langchain-text-splitters \
 rank_bm25 scikit-learn langchain-google-genai langchain-huggingface

import warnings
warnings.filterwarnings('ignore')
print('✅ !')

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
%%time
import os, json, hashlib, random, asyncio
import numpy as np

try:
 from google.colab import userdata
 os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
 print('✅ API Key Colab Secrets')
except Exception:
 api_key = input('🔑 Gemini API Key: ')
 os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

from google import genai
try:
 client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
 resp = client.models.generate_content(model='gemini-2.5-flash', contents=' OK')
 print(f'✅ API : {(resp.text.strip() if resp.text else '( output)')}')
except Exception as e:
 print(f'❌ API Error: {e}')

### + VectorDB (re-use Day 1-2)

In [ ]:
%%time
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')

documents = {
 'healthcare': ' AI X-ray 30 5 Deep Learning 95% ',
 'banking': ' KADE AI Assistant RAG 5 30 ',
 'education': ' RAG - PDF 500 vector embeddings multilingual model ',
 'agriculture': ' AI 92% 40% Computer Vision CNN train 50000 ',
 'rag': ' RAG 3 Retriever VectorDB Reader Generator hallucination LLM ',
 'embedding': 'Text Embedding vector multilingual-e5-large 100 vector 1024 vector Cosine Similarity ',
 'kmitl': ' (KMITL) AI Smart Campus IoT sensor Machine Learning 25% NLP RAG AI Tutor ',
 'nlp': 'NLP Word Segmentation PyThaiNLP Tokenizer ',
 'vectordb': 'Qdrant Vector Database ANN vectors Cosine Dot Product L2 Distance payload filter metadata ',
}

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
all_chunks, all_sources = [], []
for src, text in documents.items():
 for chunk in splitter.split_text(text):
 all_chunks.append(chunk)
 all_sources.append(src)

passages = [f'passage: {c}' for c in all_chunks]
embeddings = embed_model.encode(passages, show_progress_bar=True)
VECTOR_SIZE = embeddings.shape[1]

qdrant = QdrantClient(':memory:')
qdrant.create_collection('thai_ai_kb',
 vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE))
points = [models.PointStruct(id=i, vector=embeddings[i].tolist(),
 payload={'text': all_chunks[i], 'source': all_sources[i]}) for i in range(len(all_chunks))]
qdrant.upsert('thai_ai_kb', points=points)

print(f'✅ Data ready: {len(all_chunks)} chunks, {len(documents)} sources')

# RAG function
def search_qdrant(query, top_k=3):
 q_vec = embed_model.encode(f'query: {query}').tolist()
 results = qdrant.query_points('thai_ai_kb', query=q_vec, limit=top_k).points
 return [{'text': r.payload['text'], 'source': r.payload['source'],
 'score': round(r.score, 4)} for r in results]

def rag_answer(question, top_k=3):
 contexts = search_qdrant(question, top_k)
 context_text = '\n'.join([c['text'] for c in contexts])
 prompt = f'""" \n\n:\n{context_text}\n\n: {question}\n:"""'
 resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
 config=genai.types.GenerateContentConfig(temperature=0.3))
 return (resp.text.strip() if resp.text else '( output)'), contexts

# Quick test
ans, ctx = rag_answer('AI ')
print(f'\n🧪 Test RAG: {ans[:80]}...')

---
## 📊 Section 3.1: RAGAS — RAG

### RAGAS ?

**RAGAS (Retrieval-Augmented Generation Assessment Suite)** = framework RAG

```
Input: question + answer + contexts (+ ground_truth)
 ↓ RAGAS 
Output: 0.0 - 1.0 metric
```

### 4 Metrics 

| Metric | | | ground_truth? |
|--------|---------|------------|:------------------:|
| **Faithfulness** | "" ? (?) | answer ↔ contexts | ❌ |
| **Answer Relevancy** | ? | answer ↔ question | ❌ |
| **Context Precision** | chunk ? | contexts ↔ ground_truth | ✅ |
| **Context Recall** | chunk ? | contexts ↔ ground_truth | ✅ |

### 🌡️ 

| | | |
|:-----:|-------|----------|
| 0.8-1.0 | 🟢 | ✅ |
| 0.6-0.8 | 🟡 | ⚠️ |
| < 0.6 | 🔴 | ❌ |

In [ ]:
%%time
# ─── Evaluation Dataset ───
# : question, answer, contexts, ground_truth

eval_questions = [
 'AI ',
 'RAG ',
 ' AI ',
 'NLP ',
 'Qdrant ',
]

eval_ground_truths = [
 ' AI X-ray 30 5 95%',
 'RAG 3 Retriever VectorDB, Reader , Generator hallucination',
 ' KADE AI Assistant RAG 5 30 ',
 ' Word Segmentation PyThaiNLP ',
 'Qdrant Vector Database ANN Cosine Dot Product L2',
]

# RAG answers + contexts
eval_answers = []
eval_contexts = []

print('🔄 answers...')
for q in eval_questions:
 ans, ctxs = rag_answer(q)
 eval_answers.append(ans)
 eval_contexts.append([c['text'] for c in ctxs])
 print(f' ✅ {q[:30]}... → {ans[:40]}...')

print(f'\n📊 Eval Dataset: {len(eval_questions)} questions')

In [ ]:
%%time
# ─── RAGAS ( Gemini LLM + Local Embeddings) ───
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd

# 1. Gemini LLM (Reasoning)
gemini_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
 model='gemini-2.5-flash',
 google_api_key=os.environ['GOOGLE_API_KEY'],
))

# 2. Local Embedding (E5) API 404
local_embeddings = LangchainEmbeddingsWrapper(
 HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-large')
)

# 3. Dataset
eval_dataset = Dataset.from_dict({
 'question': eval_questions,
 'answer': eval_answers,
 'contexts': eval_contexts,
 'ground_truth': eval_ground_truths,
})

# 4. RAGAS
try:
 result = evaluate(
 dataset=eval_dataset,
 metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
 llm=gemini_llm,
 embeddings=local_embeddings,
 )

 # : result result.scores
 final_data = []
 if hasattr(result, 'scores'):
 # scores list to_pandas 
 if isinstance(result.scores, list):
 final_data = result.scores
 else:
 final_data = result.scores.to_pandas()
 elif isinstance(result, list):
 final_data = result
 else:
 final_data = [result]

 scores_df = pd.DataFrame(final_data)

 # Metric
 avg_scores = scores_df.mean(numeric_only=True)
 for metric, score in avg_scores.items():
 level = '🟢' if score >= 0.8 else ('🟡' if score >= 0.6 else '🔴')
 print(f' {level} {metric}: {score:.4f}')

except Exception as e:
 print(f'⚠️ RAGAS error: {e}')
 print('💡 error Section 3.3 (LLM-as-Judge) ')

### 💡 
- **Faithfulness ** = Agent 
- **Context Precision ** = chunk → `chunk_size`, `top_k`
- **Context Recall ** = → `top_k` embedding

> 🎯 : metric **≥ 0.7** production

### 🧪 3.1
1. 3 + ground_truth
2. RAGAS → metric ?
3. 

In [ ]:
# 🧪 : RAGAS
# 💡 Hint:
# 1. question + ground_truth list
# 2. rag_answer() answer + contexts
# 3. Dataset.from_dict() evaluate()


---
## 🔬 Section 3.2: Evaluation Dataset 

### ?

- ground_truth → (5 = 30 )
- LLM chunk → 50 2 

```
Chunk: " AI X-ray 95%"
 ↓ Gemini 
Q: " AI X-ray?"
A: " 95%"
```

In [ ]:
%%time
# ─── Auto-generate Q&A chunks ───
def generate_qa_from_chunk(chunk_text):
 prompt = f''' 1 + 
: {chunk_text}

 JSON: {{"question": "...", "answer": "..."}}
 '''
 try:
 resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
 config=genai.types.GenerateContentConfig(temperature=0.3, response_mime_type='application/json'))
 return json.loads(resp.text)
 except:
 return None

# 8 chunks ( chunk source)
auto_qa = []
seen_sources = set()
for i, chunk in enumerate(all_chunks):
 src = all_sources[i]
 if src in seen_sources: continue
 seen_sources.add(src)
 qa = generate_qa_from_chunk(chunk)
 if qa:
 qa['source'] = src
 auto_qa.append(qa)
 print(f' ✅ [{src}] Q: {qa["question"][:50]}...')

print(f'\n📊 {len(auto_qa)} Q&A pairs')

### 💡 Observations
- The LLM can generate questions about 10x faster than doing it manually.
- But you still need to **check quality** — some questions may be too broad.
- Use this as a **starting point**, then refine manually.

> 🎯 Production pattern: auto-generate 100 questions → review manually → keep the best 50.



### 🧪 Exercise 3.2
Generate auto Q&A pairs from all chunks, then review their quality.



In [ ]:
# 🧪 
# 💡 Hint: all_chunks Q&A generate_qa_from_chunk()


---
## 🧪 Section 3.3: Agent

### ?

| | | |
|--------|----------|--------|
| **Tool Selection** | tool ? | BMI → calculate_bmi |
| **Response Quality** | ? | LLM-as-Judge |
| **Edge Cases** | ? | , |
| **Guardrails** | ? | |

In [ ]:
# ─── Agent ───
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

def search_knowledge(query: str) -> dict:
 """ AI 
 Args:
 query: 
 """
 results = search_qdrant(query, top_k=3)
 return {'status': 'success', 'results': results}

def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
 """ BMI
 Args:
 weight_kg: 
 height_cm: 
 """
 h = height_cm / 100
 bmi = weight_kg / (h ** 2)
 return {'bmi': round(bmi, 1)}

test_agent = Agent(
 name='test_agent', model='gemini-2.5-flash',
 instruction=''' AI 
 - search_knowledge AI 
 - calculate_bmi BMI
 - 
 - ''',
 tools=[search_knowledge, calculate_bmi]
)

async def test_agent_call(agent, msg):
 runner = InMemoryRunner(agent=agent, app_name=agent.name)
 session_id = f's_{id(agent)}_{id(msg)}'
 await runner.session_service.create_session(
 app_name=agent.name, user_id='tester', session_id=session_id
 )
 content = types.Content(role='user', parts=[types.Part.from_text(text=msg)])
 response, tools = '', []
 async for event in runner.run_async(user_id='tester', session_id=session_id, new_message=content):
 if event.content and event.content.parts:
 for p in event.content.parts:
 if p.text: response = p.text
 if p.function_call: tools.append(p.function_call.name)
 return response, tools

print('✅ Test Agent ')

In [ ]:
# ─── Test Suite: Tool Selection ───
test_cases = [
 {'input': ' 70 175 BMI?', 'expected_tool': 'calculate_bmi'},
 {'input': 'AI ', 'expected_tool': 'search_knowledge'},
 {'input': '', 'expected_tool': None},
 {'input': 'Qdrant ', 'expected_tool': 'search_knowledge'},
 {'input': '', 'expected_tool': None},
]

print('═' * 60)
print('🧪 Test Suite: Tool Selection')
print('═' * 60)

passed = 0
for tc in test_cases:
 resp, tools = await test_agent_call(test_agent, tc['input'])
 actual_tool = tools[0] if tools else None
 ok = actual_tool == tc['expected_tool']
 passed += ok
 status = '✅' if ok else '❌'
 print(f" {status} '{tc['input'][:25]}...' → expected={tc['expected_tool']}, got={actual_tool}")

print(f'\n📊 Pass rate: {passed}/{len(test_cases)} ({passed/len(test_cases)*100:.0f}%)')

In [ ]:
%%time
# ─── LLM-as-Judge () ───
def llm_judge(question, answer, max_score=5):
 prompt = f''' 1-{max_score} ():
- : ( 2 )
- : ( = 1 )
- : bullet points 5 ( = 1 )
- : 
- : "" ( = 1 )

: {question}
: {answer}

 JSON: {{"score": 1-{max_score}, "reason": ""}}'''
 try:
 resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
 config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
 return json.loads(resp.text)
 except:
 return {'score': 0, 'reason': 'error'}

# — ( + + edge case)
print('═' * 60)
print('🧪 LLM-as-Judge: ')
print('═' * 60)

judge_questions = [
 'AI ',
 ' AI vs ',
 ' AI IPO ',
]

total_score = 0
for q in judge_questions:
 ans, _ = rag_answer(q)
 verdict = llm_judge(q, ans)
 total_score += verdict['score']
 stars = '⭐'*verdict['score'] + '☆'*(5-verdict['score'])
 print(f' {stars} {verdict["score"]}/5 | {q[:30]}... → {verdict["reason"]}')

avg = total_score / len(judge_questions)
print(f'\n📊 Average: {avg:.1f}/5.0')


### 💡 
- **Tool Selection test** = Agent tool (deterministic)
- **LLM-as-Judge** = LLM ()
- → "" ""

> ⚠️ LLM-as-Judge 100% — screening tool human review

### 🧪 3.3
 test cases 5 edge cases (, )

In [ ]:
# 🧪 
# 💡 Hint: test_cases test_agent_call()


---
## ⚡ Section 3.4: RAG Pipeline

### ?

| | Parameter | |
|---------|----------|----------|
| Chunking | `chunk_size`, `overlap` | Context Precision |
| Retrieval | `top_k`, `alpha` | Context Recall |
| Generation | `temperature`, prompt | Faithfulness, Relevancy |

### A/B Experiment
```
Config A: chunk=100, top_k=3
Config B: chunk=200, top_k=5 ← ?
Config C: chunk=300, top_k=3
```

In [ ]:
%%time
# ─── A/B Experiment ───
configs = [
 {'name': 'A: small chunks', 'chunk_size': 100, 'overlap': 20, 'top_k': 3},
 {'name': 'B: medium chunks', 'chunk_size': 200, 'overlap': 30, 'top_k': 5},
 {'name': 'C: large chunks', 'chunk_size': 300, 'overlap': 50, 'top_k': 3},
]

test_q = 'AI '
test_gt = ' AI X-ray 95%'

print('═' * 60)
print('🧪 A/B Experiment: Config')
print('═' * 60)

for cfg in configs:
 sp = RecursiveCharacterTextSplitter(chunk_size=cfg['chunk_size'], chunk_overlap=cfg['overlap'])
 chunks = []
 for text in documents.values():
 chunks.extend(sp.split_text(text))
 
 # Embed + search
 vecs = embed_model.encode([f'passage: {c}' for c in chunks])
 q_vec = embed_model.encode(f'query: {test_q}')
 from sklearn.metrics.pairwise import cosine_similarity
 sims = cosine_similarity([q_vec], vecs)[0]
 top_idx = np.argsort(sims)[-cfg['top_k']:][::-1]
 top_chunks = [chunks[i] for i in top_idx]
 top_scores = [sims[i] for i in top_idx]
 
 # Check if ground truth info is in retrieved chunks
 gt_found = any('' in c or '95%' in c for c in top_chunks)
 
 print(f'\n📋 {cfg["name"]}: {len(chunks)} chunks')
 print(f' Top scores: {[round(s,3) for s in top_scores]}')
 print(f' Ground truth found: {"✅" if gt_found else "❌"}')
 print(f' Best chunk: {top_chunks[0][:60]}...')

### 💡 
- **chunk ** (100) → chunks 
- **chunk ** (300) → noise 
- **top_k ** → recall precision 

> 🎯 config — **** 

### 🧪 3.4
 3 configs → config 

In [ ]:
# 🧪 : configs 
# 💡 Hint: configs list 


---
## ✍️ Section 3.5: Prompt Engineering for Agent

### Techniques

| Technique | Example | Reduce Hallucination? |
|--------|---------|:-----------------:|
| **Role** | "You are an expert..." | ⭐ |
| **Few-shot** | Provide example Q&A | ⭐⭐ |
| **Chain-of-Thought** | "Think step by step..." | ⭐⭐⭐ |
| **Guardrails** | "Do not guess; if unsure, say so clearly" | ⭐⭐⭐ |
| **Output Format** | "Respond in 3 bullet points" | ⭐ |



In [ ]:
# ─── Before vs After Prompt ───
prompt_before = ''
prompt_after = ''' AI 

:
1. search_knowledge 
2. 
3. [healthcare] [banking]
4. ""
5. bullet points 5 '''

agent_v1 = Agent(name='v1', model='gemini-2.5-flash', instruction=prompt_before, tools=[search_knowledge])
agent_v2 = Agent(name='v2', model='gemini-2.5-flash', instruction=prompt_after, tools=[search_knowledge])

# — instruction 
hard_test_questions = [
 ' AI vs ',
 ' AI ',
 ' NLP ',
]

print('═' * 60)
print('🧪 Before vs After Prompt ()')
print('═' * 60)

v1_total, v2_total = 0, 0
for q in hard_test_questions:
 r1, _ = await test_agent_call(agent_v1, q)
 r2, _ = await test_agent_call(agent_v2, q)

 j1 = llm_judge(q, r1)
 j2 = llm_judge(q, r2)
 v1_total += j1['score']
 v2_total += j2['score']

 print(f'\n❓ {q}')
 print(f' V1: {j1["score"]}/5 — {j1["reason"]}')
 print(f' V2: {j2["score"]}/5 — {j2["reason"]}')

v1_avg = v1_total / len(hard_test_questions)
v2_avg = v2_total / len(hard_test_questions)
print(f'\n{"═"*60}')
print(f'📊 : V1 avg = {v1_avg:.1f}/5 | V2 avg = {v2_avg:.1f}/5')
diff = v2_avg - v1_avg
if diff > 0:
 print(f'📈 V2 V1 +{diff:.1f} — instruction !')
elif diff < 0:
 print(f'📉 V1 V2 — instruction ')
else:
 print(f'➡️ — ')


### 💡 
- Instruction → **, , **
- Instruction → **, , reference**
- **Guardrails** ("") hallucination 

> 🎯 Prompt Engineering = ** **

### 🧪 3.5
 instruction 3 → LLM-as-Judge → 

In [ ]:
# 🧪 
# 💡 Hint: agent_v3 instruction 


---
## 🚀 Section 3.6: Capstone Project ⭐

### Combine everything from all 3 days

```
Day 1: Prepare data → Chunk → Embed → VectorDB
Day 2: Build Agent → Tools → RAG Agent → Multi-Agent
Day 3: Measure with RAGAS → test → improve → measure again
```

### Task
1. **Build a RAG Pipeline** from the assigned data.
2. **Measure quality** with RAGAS + LLM-as-Judge.
3. **Improve performance** (chunk_size, prompt, top_k).
4. **Show Before/After results**.



In [ ]:
# ─── Capstone: Before → After ───
print('═' * 60)
print('🚀 Capstone: RAG Pipeline')
print('═' * 60)

# ( + + edge case) 
capstone_questions = [
 'AI ',
 ' embedding model ',
 '- RAG fine-tuning',
 'Elon Musk ',
 ' use case AI ',
]

# Step 1: baseline (rag_answer )
print('\n📊 Step 1: Baseline (RAG )')
baseline_scores = []
for q in capstone_questions:
 ans, _ = rag_answer(q)
 verdict = llm_judge(q, ans)
 baseline_scores.append(verdict['score'])
 stars = '⭐'*verdict['score'] + '☆'*(5-verdict['score'])
 print(f' {stars} {verdict["score"]}/5 | {q[:35]}...')
baseline_avg = sum(baseline_scores) / len(baseline_scores)
print(f' → Baseline Average: {baseline_avg:.1f}/5.0')

# Step 2: ( Agent + instruction )
print('\n📊 Step 2: After Optimization (Agent + Good Prompt)')
improved_scores = []
for q in capstone_questions:
 r, _ = await test_agent_call(agent_v2, q)
 verdict = llm_judge(q, r)
 improved_scores.append(verdict['score'])
 stars = '⭐'*verdict['score'] + '☆'*(5-verdict['score'])
 print(f' {stars} {verdict["score"]}/5 | {q[:35]}...')
improved_avg = sum(improved_scores) / len(improved_scores)
print(f' → Improved Average: {improved_avg:.1f}/5.0')

# Step 3: 
print(f'\n{"═"*60}')
print(f'📊 Before → After')
print(f'{"═"*60}')
print(f' Baseline: {"⭐" * round(baseline_avg)} {baseline_avg:.1f}/5.0')
print(f' Improved: {"⭐" * round(improved_avg)} {improved_avg:.1f}/5.0')
diff = improved_avg - baseline_avg
if diff > 0:
 print(f' 📈 +{diff:.1f} !')
elif diff < 0:
 print(f' 📉 {diff:.1f} — baseline')
else:
 print(f' ➡️ — ')


### 💡 
- **Prompt Engineering** 
- **Chunk size** — 
- **-** — 

> 🎯 "What gets measured gets improved" — 

### 🧪 3.6
1. 2 (prompt + chunk_size top_k)
2. Before/After
3. 

In [ ]:
# 🧪 Capstone: 
# 💡 Hint:
# 1. chunk_size → re-embed → re-search → 
# 2. instruction → agent → 
# 3. top_k → rag_answer( ,top_k=5) → 


---
## 🎯 : 3 

| Day | | |
|-----|--------|-------------|
| **Day 1** | Data Engineering Pipeline | Qdrant, E5-large, BM25 |
| **Day 2** | Agent Building | Google ADK, Tools, Multi-Agent |
| **Day 3** | Evaluation & Optimization | RAGAS, LLM-as-Judge |

### Skills 
- ✅ RAG Pipeline 
- ✅ Agent 
- ✅ + 
- ✅ 

### 🛣️ ?
- **Production**: Deploy Cloud (GCP, AWS) + Database session
- **Advanced**: Fine-tune embedding domain 
- **Scale**: Qdrant Cloud + horizontal scaling
- **Monitor**: Log + alert 

---

**🙏 3 !**

> "The best way to learn AI is to build with AI."